# Round 7 final test — resume with documented precision correction

The original freeze and completed outputs remain unchanged. Read `ROUND7_FINAL_PRECISION_NOTE.md`. This notebook resumes only incomplete work and validates the numerical-reporting addendum. It skips all scoring once final evaluation is complete.

In [ ]:
from pathlib import Path
import os, subprocess, sys

project = Path(r"C:\sc4001\prjJ\SC4001-AI-Image-Detection")
env = os.environ.copy()
env["PYTHONDONTWRITEBYTECODE"] = "1"
env["PYTHONIOENCODING"] = "utf-8"
env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

def run_script(script, *args):
    command = [sys.executable, "-u", str(project / "scripts" / script), *args]
    process = subprocess.Popen(command, cwd=project, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if code:
        raise subprocess.CalledProcessError(code, command)

print("Project:", project)
print("Kernel:", sys.executable)


## 1. Retain or create the final freeze
Do not edit the frozen scripts, source cells, models or settings after this step. Notebook outputs and execution counters are excluded from the source hash.

In [ ]:
freeze_path = project / "results/round7/final_freeze.json"
if not freeze_path.exists():
    run_script("round7_final_run.py", "--freeze")
else:
    print("Existing final freeze retained:", freeze_path)


## 2. Evaluate once / resume incomplete work
Keep this cell running. Progress is visible. Interrupting stops its child process; rerun this cell to continue validated batches/chunks. If completed, it skips scoring.

In [ ]:
complete = project / "results/round7/final_test/complete.json"
if complete.exists():
    print("Final evaluation is already complete. Skipping all scoring; use the next verification cell.")
else:
    subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name(0))"],
                   cwd=project, env=env, check=True)
    run_script("round7_final_precision_resume.py", "--resume")


## 3. Verify saved results (read-only inference policy)
Recomputes metrics and checks frozen hashes without loading images or models. The optional report is created exclusively or checked for identical contents.

In [ ]:
run_script("round7_final_precision_resume.py", "--verify")

## 4. Read final tables
Untouched means not used for Round 7 development; the independent-pilot overlap disclosure still applies. The main summary is the equal-weight four-generator untouched macro.

In [ ]:
from IPython.display import display, Markdown
import pandas as pd

final = project / "results/round7/final_test"
display(Markdown((final / "SUMMARY.md").read_text(encoding="utf-8")))
metrics = pd.read_csv(final / "metrics_mean_sd.csv")
paired = pd.read_csv(final / "paired_differences.csv")
display(metrics[(metrics["generator"] == "untouched_macro") & (metrics["metric"] == "auroc")])
display(paired[(paired["generator"] == "untouched_macro") & (paired["metric"] == "auroc")])
